# 00 · Environment and fundamentals

This series walks through every flow of the local Minimum Viable Dataspace (MVD) and explains, call by call, **what each request does, why it is needed and what comes back**.
Run the notebooks in order: each one saves the ids it creates in `tutorial_state.json` for the next.

| # | Notebook | Block | Concepts introduced |
|---|---|---|---|
| 00 | `00_environment.ipynb` | — | dataspace, EDC runtimes, the four APIs, JSON-LD, DIDs |
| 01 | `01_identity_onboarding.ipynb` | A | Verifiable Credentials, issuer/holder/verifier, DCP, revocation |
| 02 | `02_publishing.ipynb` | B | asset, data address, ODRL policy, policy scopes, contract definition |
| 03 | `03_discovery.ipynb` | C | DSP, DCAT catalog, offers, federated catalog |
| 04 | `04_negotiation.ipynb` | D | DSP negotiation protocol, state machine, agreement |
| 05 | `05_transfer.ipynb` | E | transfer process, pull vs push, EDR, policy monitor |
| 06 | `06_telemetry_billing.ipynb` | F | consumption records, event broker, billing |
| 07 | `07_inspection.ipynb` | G | where the state lives: Vault, SQL stores, logs |

**Before starting**, the MVD must be up:

```bash
cd ging_deploy && source ./rc.sh && ./deploy-local-mvd.sh up
```

Further reading: [FLOWS.md](../FLOWS.md) (flow ids such as A4, D1), [DIAGRAMS.md](../DIAGRAMS.md) (sequence diagrams and glossary), [DEPLOYMENT-REPORT.md](../DEPLOYMENT-REPORT.md) (technologies and configuration).

## Concept: what a dataspace is

A **dataspace** lets organisations share data without handing it to a central platform. Each organisation keeps its data, and
access happens **peer to peer, under rules (policies) the owner sets**. Two kinds of things make that work:

- **Trust**: knowing who the other party really is, and which attributes it has (member of the dataspace, works in the "route" domain…).
  Here that is provided by **decentralised identifiers (DIDs)** and **Verifiable Credentials (VCs)** signed by a trusted authority.
- **Agreements**: before any byte moves, the consumer and the provider negotiate a **contract** that records which policy applies.

Our MVD has three participants:

| Participant | Role |
|---|---|
| **authority** | Operates the dataspace: issues credentials (who is a member, which domain), runs the federated catalog and billing |
| **provider** | Owns data (a sample HTTP API) and offers it under policies |
| **consumer** | Finds data, negotiates contracts and reads the data |

## Concept: EDC and its runtimes

**Eclipse Dataspace Components (EDC)** is a Java framework for building the pieces of a dataspace. Each piece is a *runtime* (one Java process, one pod here) assembled from *extensions*.
The **DSE** extensions are Amadeus' additions on top of EDC (credential-based catalog filtering, agreement retirement, telemetry, custom policy functions…).

| Runtime | Per | Job |
|---|---|---|
| **Control plane** | provider, consumer | The connector's brain: assets, policies, offers, contract negotiation, transfer processes. Speaks **DSP** with other control planes |
| **Data plane** | provider, consumer | Moves the actual bytes once a transfer is agreed |
| **Identity Hub** | every participant | The participant's **wallet**: its DID document, its keys, its credentials, and a token service (STS) |
| **Issuer Service** | authority | Signs Verifiable Credentials for the participants |
| **Federated Catalog** | authority | Collects every participant's catalog into one searchable cache |
| **Telemetry services** | authority (+ an agent per participant) | Usage records and billing reports |

The **control plane / data plane split** is central to EDC. Negotiating *whether* data may move (control) is kept apart from *moving* it (data), so the data path can be scaled or replaced independently.

## Concept: the four kinds of API

Every EDC runtime exposes several HTTP APIs on different ports. Knowing which is which explains most of the URLs in this series:

| API | Who calls it | Example in this series | Port |
|---|---|---|---|
| **Management API** | *you*, the participant's own operator | `/provider/cp/management/v3/assets` | 8181 |
| **Protocol API (DSP)** | *other connectors* | `http://provider-controlplane:8282/api/dsp/2025-1` (we never call it directly) | 8282 |
| **Control API** | the participant's own runtimes (control plane ↔ data plane) | internal | 8383 |
| **Public / data API** | data consumers | `/consumer/dp/data` | 8181 / 8282 |

Plus the **Identity API** of the wallets (`/<p>/ih/identity/...`) and the **Issuer admin API** (`/authority/is/issueradmin/...`).

So the pattern is always: **you tell your own connector what to do through its Management API, and it talks to the other connector over DSP.**
You never call the other side's DSP endpoint yourself.

Everything goes through the ingress at `http://localhost` with a path prefix per participant (`/authority`, `/provider`, `/consumer`).
None of these APIs require authentication in this local setup.

## Concept: JSON-LD in one minute

EDC request and response bodies are **JSON-LD**: JSON whose keys are URIs, so different vocabularies can be mixed without clashes.

```json
{"@context": {"@vocab": "https://w3id.org/edc/v0.0.1/ns/"}, "@id": "my-asset", "@type": "Asset", "properties": {"name": "x"}}
```

- `@context` says how to turn short keys into full URIs. With `@vocab`, `name` means `https://w3id.org/edc/v0.0.1/ns/name`.
- `@id` is the object's identifier and `@type` its type.
- Prefixes such as `odrl:`, `dcat:`, `dct:` or `dse-policy:` abbreviate other vocabularies: policies (ODRL), catalogs (DCAT, Dublin Core) and the DSE policy functions.

When a request is rejected with an odd validation error, the cause is almost always a missing `@context`: a key was expanded into the wrong namespace.
`dataspace.py` defines the two contexts used in this series: `ds.EDC_CONTEXT` and `ds.MANAGEMENT_CONTEXT` (which also brings in ODRL).

## The helper module

[`dataspace.py`](dataspace.py) only contains plumbing. The one thing to know: **`ds.get / ds.post / ds.delete` print `METHOD path -> status`** and return the
`requests` response. You can then read `response.status_code`, `response.headers` and `response.json()`.
`expect={...}` makes the call fail loudly if the status is not one of those listed.

In [1]:
import dataspace as ds

import uuid

print("ingress:      ", ds.BASE_URL)
print("kube context: ", ds.KUBE_CONTEXT)
for participant in ds.PARTICIPANTS:
    print(f"{participant:10} DID: {ds.did(participant)}")

ingress:       http://localhost
kube context:  kind-dse-cluster
authority  DID: did:web:authority-identityhub%3A8383:api:did
provider   DID: did:web:provider-identityhub%3A8383:api:did
consumer   DID: did:web:consumer-identityhub%3A8383:api:did


## A run id for this pass through the tutorial

Every asset created by these notebooks gets this suffix, so the series can be run again on the same deployment without id clashes.

In [2]:
run_id = uuid.uuid4().hex[:6]
ds.remember(run_id=run_id)

   remembered run_id = 42930a


## Is everything running?

**What:** `kubectl get pods` on the local kind cluster, with pods grouped by name prefix.

**Why:** each runtime described above is one pod. Init jobs (`*-db-init`, `*-vault-*-job`) are one-off pods that prepared databases and secrets.

**Expect:** long-running pods `Running` with all containers ready (`1/1`); jobs `Completed`.

In [3]:
pods_by_group = {"authority": [], "provider": [], "consumer": [], "shared infra": []}
problems = []

for line in ds.kubectl("get", "pods", "--no-headers").splitlines():
    name, ready, status = line.split()[:3]
    group = name.split("-")[0] if name.split("-")[0] in pods_by_group else "shared infra"
    pods_by_group[group].append(f"{name:55} {ready:5} {status}")
    if status not in ("Running", "Completed"):
        problems.append(name)

for group, pods in pods_by_group.items():
    print(f"== {group}")
    for pod in pods:
        print("  ", pod)
print("\nproblems:", problems or "none")

== authority
   authority-db-init-d569t                                 0/1   Completed
   authority-federatedcatalog-758df67bfc-4rghg             1/1   Running
   authority-federatedcatalogfilter-9df59574f-8pt89        1/1   Running
   authority-identityhub-5f5b78fc6b-zkxjb                  1/1   Running
   authority-issuerservice-7765cf9784-nrdrx                1/1   Running
   authority-telemetrycsvmanager-69c586444d-b7m7j          1/1   Running
   authority-telemetryservice-5c4c8867c6-692nl             1/1   Running
   authority-telemetrystorage-f4476567f-qgk5g              1/1   Running
   authority-vault-0                                       1/1   Running
== provider
   provider-backend-6db9856454-2n4q5                       1/1   Running
   provider-controlplane-6d9d997dbf-2vxk4                  1/1   Running
   provider-dataplane-649679959b-cqkfz                     1/1   Running
   provider-db-init-tgpdk                                  0/1   Completed
   provider-identityhu

## The ingress only answers for `Host: localhost`

**What:** the same request twice, once with `Host: localhost` and once with a nip.io host name.

**Why:** every Ingress rule in this deployment is declared for the host `localhost`. nginx picks the rule by Host header *before* looking at the path.

**Expect:** 200, then 404.

In [4]:
with_localhost = ds.get(f"{ds.BASE_URL}/provider/ih/did/did.json")
with_nip_io = ds.get("http://127.0.0.1/provider/ih/did/did.json", headers={"Host": "provider.127.0.0.1.nip.io"})

GET    /provider/ih/did/did.json  ->  200
GET    http://127.0.0.1/provider/ih/did/did.json  ->  404


## Concept: DIDs and DID documents

A **DID** (decentralised identifier) names a participant without a central registry. This MVD uses the **`did:web`** method:
the DID encodes a web address where its **DID document** can be downloaded.

```
did:web:provider-identityhub%3A8383:api:did   →   http://provider-identityhub:8383/api/did/did.json
        └────────── host:port ────────┘└path┘       (%3A is an escaped ":")
```

The DID document says two things:

- **verificationMethod**: the participant's public key(s). Anyone can check a signature made by this participant with it.
- **service**: endpoints where other parties reach this participant's services. For example, where to ask for its credentials (`CredentialService`) or where its connector speaks DSP (`DSPMessaging`).

The DID is also the participant's id everywhere in EDC (`EDC_PARTICIPANT_ID`). The host is an in-cluster name, so these DIDs only resolve inside the cluster.

## A9 · Fetch the three DID documents

**What:** `GET /<p>/ih/did/did.json` on each identity hub.  **Returns:** the DID document as JSON.

In [5]:
for participant in ds.PARTICIPANTS:
    did_document = ds.get(f"{ds.BASE_URL}/{participant}/ih/did/did.json").json()

    print("   id: ", did_document["id"])
    for key in did_document.get("verificationMethod", []):
        jwk = key["publicKeyJwk"]
        print(f"   key: {key['id']}  ({key['type']}, {jwk['kty']} {jwk.get('crv', '')})")
    for service in did_document.get("service", []):
        print(f"   service {service['type']:30} {service['serviceEndpoint']}")
    print()

GET    /authority/ih/did/did.json  ->  200
   id:  did:web:authority-identityhub%3A8383:api:did
   key: did:web:authority-identityhub%3A8383:api:did#my-key  (JsonWebKey2020, EC P-256)
   service IssuerService                  http://authority-issuerservice:8282/api/issuance/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ
   service FederatedCatalogService        http://authority-federatedcatalog.default.svc.cluster.local:8383/api/catalog/v1alpha/catalog/query
   service TelemetryServiceCredential     http://authority-telemetryservice:8181/api/credential
   service CredentialService              http://authority-identityhub.default.svc.cluster.local:8282/api/credentials/v1/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ
   service FederatedCatalogFilterService  http://authority-federatedcatalogfilter.default.svc.cluster.local:8383/api/catalogfilter/filter

GET    /provider/ih/did/did.json  ->  200
   id:  did:web:provider-identit

Reading the output:

- Every participant has one key `#my-key`: an **EC P-256** key (`JsonWebKey2020`). Its private half sits in the participant's Vault.
- Provider and consumer advertise `CredentialService` (their wallet) and `DSPMessaging` (their connector).
- The authority additionally advertises `IssuerService` (where to request credentials), `FederatedCatalogService` / `FederatedCatalogFilterService` and `TelemetryServiceCredential`.
  The other runtimes **discover** these endpoints by resolving the authority's DID instead of having them configured.

Next: **01_identity_onboarding.ipynb**